# PyTorch Lightning fine-tuning template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a surya model to implement DS application.

It focuses on the concept of defining a modified Surya model, loading its weigths, and using a PyTorch lightning training loop to train it

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**

It doesn't require having seen the baselines template, but they are meant to complement each other.  **_In fact they are on purpose almost identical!!!_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
torch.set_float32_matmul_precision('medium')



## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [ ]:
# The config is the single source of truth. load_radioburst_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.radioburst.configs import load_radioburst_config

cfg = load_radioburst_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [6]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


[assets] surya.366m.v1.pt not found — downloading from nasa-ibm-ai4science/Surya-1.0 ...


surya.366m.v1.pt:   0%|          | 0.00/1.80G [00:00<?, ?B/s]

[assets] Saved to /home/jovyan/surya_workshop/downstream_apps/radioburst/assets/surya.366m.v1.pt
Loaded scalers for 13 channels.


### Typed configuration, and how to extend it for your own task

`load_radioburst_config()` reads `configs/config_script.yaml` and returns a typed `TrainingConfig`.
This notebook and `3_finetune_template_1D.py` call the same function on the same file, so
there is no notebook-versus-script divergence to reason about.

| YAML section | Access in Python | Dataclass |
|---|---|---|
| `data:` | `cfg.data.*` | `RadioBurstDataConfig` (this app) |
| `model:` | `cfg.model.*` | `ModelConfig` |
| `model.lora_config:` | `cfg.model.lora_config.*` | `LoraAdapterConfig` |
| `model.time_embedding:` | `cfg.model.time_embedding.*` | `TimeEmbeddingConfig` |
| `training:` | `cfg.learning_rate`, `cfg.batch_size`, … | `TrainingConfig` (flat) |
| `output:` | `cfg.output.*` | `OutputConfig` |
| `logging:` | `cfg.wandb_project`, `cfg.wandb_entity` | `TrainingConfig` (flat) |

**Everything except `RadioBurstDataConfig` lives in `workshop_infrastructure/configs.py`** and is
shared by every downstream app. When you fork the template you do not copy that file. You
subclass `DataConfig` with your task's fields and bind `load_config` to it — this app's
`downstream_apps/radioburst/configs.py` follows the same pattern:

```python
@dataclass
class RadioBurstDataConfig(DataConfig):
    ds_radioburst_folder_path: str = ""
    ds_radioburst_index_file: str = ""
    ds_time_column: str = "window_start"
    ds_time_tolerance: str = "1h"
    ds_match_direction: str = "forward"
    ds_spectra_column: str = "window_start_file"
    ds_spectra_template_file: str = ""
    ds_diagnostics_columns: list[str] | None = None
    PATH_FIELDS = DataConfig.PATH_FIELDS + ("ds_radioburst_folder_path",)   # resolve it like a path

load_radioburst_config = partial(load_config, data_cls=RadioBurstDataConfig)
```

Unknown keys are rejected rather than silently dropped: if you add a key to the YAML before
adding the field, you get an error naming the key and listing the valid ones.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additonal parameters relevant to the downstream application.  Here we focus in particular to the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is normalization.  Here we use a log normalization on xray flux that will act as the output target.  Making log10(xray_flux) strictly positive and having 66% of its values between 0 and 1

In this case we will define both a training and a validation dataset using the indices pointed at in the config

**_Important:  In this notebook we sets max_number_of_samples=6 to potentially avoid going through the whole dataset as we explore it.  Keep in mind this for the future in case the database seems smaller than you expect_**


In [8]:
from downstream_apps.radioburst.datasets.radioburst_dataset import RadioBurstDSDataset

### Normalize the spectra target

The fine-tuned model regresses the full radio spectrogram, so the target's scale matters. Raw flux is strictly positive and spans about six orders of magnitude (roughly `1e-15` to `1e-9`): an MSE on those values is vanishingly small, which gives the optimizer essentially no gradient, and the brightest bins would dominate anyway.

We therefore use the same recipe the flare template applies to X-ray flux: take `log10`, shift so the minimum is 0, and divide by twice the standard deviation. The dataset applies `spectra_transform` once to the whole catalog before splitting, so the training and validation sets share identical statistics.

**Missing values.** About 0.1% of spectrogram cells are `NaN` — scattered gaps, all in burst windows. We don't fill them in (that would invent data): the transform ignores them when computing its statistics, and the loss skips them.

In [ ]:
import numpy as np
import pandas as pd


def log10_spectra_transform(raw: pd.Series) -> pd.Series:
    """log10, shift so the minimum is 0, scale by 2 * std. Statistics are global over all bins.

    Missing cells stay NaN (nan-aware statistics); the loss skips them.
    """
    log_spectra = raw.apply(lambda spectrum: np.log10(spectrum))
    stacked = np.stack(log_spectra.to_list())
    minimum = np.nanmin(stacked)
    scale = 2 * np.nanstd(stacked - minimum)
    return log_spectra.apply(lambda spectrum: ((spectrum - minimum) / scale).astype(np.float32))

In [ ]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the radio-burst-specific arguments are passed here, which is
# exactly the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    RadioBurstDSDataset,
    scalers=scalers,
    num_workers=8,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=6,
    ds_radioburst_folder_path = cfg.data.ds_radioburst_folder_path,
    ds_radioburst_index_file=cfg.data.ds_radioburst_index_file,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
    ds_spectra_column=cfg.data.ds_spectra_column,
    ds_diagnostics_columns=cfg.data.ds_diagnostics_columns,
    ds_spectra_template_file=cfg.data.ds_spectra_template_file,
    spectra_transform=log10_spectra_transform,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [ ]:
# Inspect a single batch to confirm shapes before building the model.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})

# The model's spectrogram output must match the target's (T, F) shape.
spectrum_shape = tuple(batch["spectra"].shape[1:])
spectra = batch["spectra"]
print(f"spectrum_shape: {spectrum_shape} | spectra range after transform: "
      f"[{spectra.nan_to_num(float('inf')).min():.2f}, {spectra.nan_to_num(float('-inf')).max():.2f}] | "
      f"missing (NaN) cells: {spectra.isnan().float().mean():.2%}")


## Initialize the HelioSpectformer model

This is the main difference beteween the notebook that trains the simple model and the one that fine-tunes Surya.  

In the case of the finetuning exercise one of the main differences between DS applications is the dimensionality of the output.  In this notebook we use a modified HelioSpectformer that projects into a 1D space. 

**_IMPORTANT: If your DS application is 2D you need to use the HelioSpectformer2D_**

### A custom head for radio bursts

Our task needs two outputs per sample: a **burst logit** (is there a burst?) and a full **`(T, F)` radio spectrogram**. `ADAPTING.md` (Step 5) says a multi-output task gets a custom class in the app's `models/` folder, so `models/finetune_burst.py` defines `HelioSpectformerBurst`, a small subclass of `HelioSpectformer1D`:

```
backbone (frozen + LoRA) → pooling → head_linear → head_unembed: Linear(1280, 1 + r)
                                                        ├── column 0 ─────► burst_logit (B, 1)
                                                        └── r coefficients ─► head_spectra_decoder: Linear(r, T·F) ─► spectra (B, T, F)
```

The decoder learns `r` basis spectrograms and each prediction is a weighted mix of them. That bottleneck keeps the head at ~2.8M parameters instead of the ~44M a flat `Linear(1280, T·F)` would need, which matters with only a few hundred training windows. Both new layers start with `head_`, so LoRA keeps them trainable.

The model returns a **logit** rather than a probability: the loss uses `binary_cross_entropy_with_logits`, which is numerically stable and, unlike plain BCE on sigmoid outputs, is allowed under `bf16-mixed` precision.

In [ ]:
from downstream_apps.radioburst.models.finetune_burst import HelioSpectformerBurst

Now the config file really comes into bear. The Spectformer has a metric ton of hyperparameters

In [ ]:
# HelioSpectformerBurst inherits HelioSpectformer1D's long list of architecture arguments,
# and all of them come straight from the model: section of the config. from_config() does
# that mapping, so the backbone can never drift out of sync with the checkpoint it is about
# to load.
#
# Arguments that are not part of ModelConfig are passed as explicit overrides: dtype, plus
# the head's spectrum_shape (read from the batch above) and spectra_rank (the number of
# learned basis spectrograms).
model = HelioSpectformerBurst.from_config(
    cfg.model,
    spectrum_shape=spectrum_shape,
    spectra_rank=32,
    dtype=cfg.dtype,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
)


## Load model weights

Here we load the pre-trained checkpoint and load the weights.  The exercise of loading follows the idea of us as many of the weights as possible.  This is accomplished through the filtered_checkpoint_state.   It checks to see if the pretrained model's layers match those of your finetuning architecture.   It also checks that all your dimensions across layers check out.   If something does not work those paramameters are left in their random initialization. 

In [ ]:
# The checkpoint was saved from HelioSpectFormer directly, so its keys are flat
# (e.g. "embedding.proj.weight"), while the fine-tuning model nests the backbone under
# "backbone.*". load_pretrained_weights() tries both spellings and reports how many
# tensors matched — a low count means the architecture does not match the checkpoint.
from workshop_infrastructure.utils import load_pretrained_weights

load_pretrained_weights(model, cfg.model.pretrained_path)


## To LoRA or not to Lora

This cell gives you two options.  On the one hand we have the classic freezing of the backbone (the initial layers of the model).   On the other hand we have the use of a LoRA.

LoRas have been a remarkable addition to our arsenal of models.   They have the advantage of keeping pretty much the entire model intact and only add broad modifications to weights as needed.

**What actually trains.** In the LoRA regime the pretrained Surya weights stay frozen; what trains is the adapters *and* the whole fine-tuning head. That second part is easy to get wrong: PEFT freezes every parameter it does not recognise as an adapter, so unless the head is explicitly handed to it as `modules_to_save`, the adapters end up fitting a **frozen, randomly initialised readout** — and the loss still goes down, so the training curve looks perfectly healthy. `apply_peft_lora()` avoids this by discovering every `head_*` attribute on the model and marking it trainable. If you add your own head layer, give it a `head_` prefix or it will be silently frozen (the helper checks this at startup and tells you what to rename).

The head is always fully trainable, because it starts from a random initialisation — LoRA's small corrections only make sense on weights that were already pretrained.

**Where the adapters go.** `fc1`/`fc2` in all ten blocks, plus `attn.qkv` and `attn.proj` in the eight attention blocks. The spectral `complex_weight`, `attn.to_dynamic_projection`, and the patch embedding are never adapted.

Surya fuses query, key and value into a single `nn.Linear(1280, 3840)`, so one adapter covers all three at once: they share the `8×1280` matrix `A` and each gets its own `1280×8` slice of `B`. Their combined rank is at most 8 — which is *not* the same as giving q, k and v three independent rank-8 adapters.

Run the cell below and check the printout. `[LoRA] Trainable head modules` should list `head_cls_token`, `head_linear`, `head_unembed` and `head_spectra_decoder`. With `spectra_rank=32` and a `(120, 288)` spectrogram, the LoRA regime should report **4,339,233** trainable parameters (1,515,520 of adapters + 2,823,713 of head), and the linear probe **2,823,713**.

In [ ]:
# Three fine-tuning regimes, all selected from the model: section of the config:
#
#   use_lora: true                          -> LoRA adapters + the whole head (default)
#   use_lora: false, freeze_backbone: true  -> linear probe: only the head trains
#   use_lora: false, freeze_backbone: false -> full fine-tuning of all 366M parameters
#
# freeze_backbone is ignored when use_lora is true: PEFT freezes everything, then
# re-enables the adapters and every head_* module.
#
# 3_finetune_template_1D.py applies exactly this logic in build_model().
if cfg.model.freeze_backbone:
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = False

if cfg.model.use_lora:
    # Prints the adapted modules and the trainable head modules it discovered.
    model = apply_peft_lora(model, cfg.model.lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


We can now test that this model manipulates a batch as expected and returns a burst logit and a predicted spectrogram.

We pass the batch to the model to transform it into our two outputs.   Note that since the backbone was trained for a different task and the head is randomly initialized, the output has no real meaning yet.  This only acts as a test that our model forward doesn't have dimension problems.

Dimension problemns are the dominant source of error in this kind of work.

Check that `burst_logit` has shape `(batch, 1)` and `spectra` has shape `(batch, T, F)`, matching `batch["spectra"]`.

In [ ]:
batch = next(iter(train_data_loader))
output = model.forward(batch)
{k: tuple(v.shape) for k, v in output.items()}

## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantitification of error, which in turn shifts the weights towards better pefrorming models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics do you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can be later use in a training script

In [ ]:
from downstream_apps.radioburst.metrics.radioburst_metrics import RadioBurstSpectraMetrics

In [ ]:
# RadioBurstSpectraMetrics matches HelioSpectformerBurst's outputs (burst_logit, spectra).
# The linear baseline in notebook 1 uses RadioBurstMetrics instead (burst_prob, peak_amp).
train_loss_metrics = RadioBurstSpectraMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same loss as train_loss — override RadioBurstSpectraMetrics.val_loss to change it.
val_loss_metrics = RadioBurstSpectraMetrics("val_loss")
train_evaluation_metrics = RadioBurstSpectraMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = RadioBurstSpectraMetrics("val_metrics")

Now they can be evaluated in our model's output and our ground truth.   First the loss that actually will backpropagate. It has two terms:

- `bce`: binary cross-entropy on the burst logit, for every window.
- `mse_spectra`: mean squared error between predicted and true spectrograms, **only for burst windows** (`burst == 1`). Quiet windows teach the classifier but not the spectrogram decoder. If a batch happens to contain no bursts, this term is 0.

In [ ]:
target = {"burst": batch["burst"], "diagnostics": batch["diagnostics"], "spectra": batch["spectra"]}
train_loss_metrics(output, target)

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it her as an example, but oftentimes is better to put the diagnostics only in the validation evaluation metrics.

Here we report the burst-classification F1 score and the Root Relative Squared Error of the spectrogram (burst windows only) https://lightning.ai/docs/torchmetrics/stable/regression/rse.html 

An RRSE below one means the prediction is better than predicting the average.  It is unlikely that this metric will be lower than one with a randomly initialized head

In [ ]:
train_evaluation_metrics(output, target)

In the validation evaluation metrics we report F1, plus both MSE and RRSE of the spectrogram

In [ ]:
validation_evaluation_metrics(output, target)

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the RadioBurstLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

**_Note that it is the same Lightning module we used for the baseline!!_**

In [ ]:
from downstream_apps.radioburst.lightning_modules.pl_simple_baseline import RadioBurstLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [ ]:
L.seed_everything(42, workers=True)

## Intialize Lightning module

Now we properly initalize the Lightning module to enable training, including passing the dictionary of metrics

In [ ]:
metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

lit_model = RadioBurstLightningModule(model, metrics, lr=cfg.learning_rate, batch_size=batch_size)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [ ]:
project_name = cfg.wandb_project
run_name = "finetune_experiment_1"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).


**Note that in this notebook we also set a mixed precision to reduce the model's footprint in memory.**

In [ ]:
max_epochs = 2

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    precision="bf16-mixed", 
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

## Fit the model

Finally we fit the model.  We pass the Lighting module, and our dataloaders.

In [ ]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and DS into an end-2-end training loop and we are ready to experiment!